# Vehicle Fuel Efficiency: Endterm Report

**Team:** Aktore, Khamza, Askhat. We predict EPA combined US MPG from vehicle specifications. This notebook reads frozen endterm artifacts and does not tune models or reopen test data. MAE in MPG is primary; RMSE in MPG and R-squared are secondary.

## Dataset, scope and leakage prevention

The EPA/FuelEconomy.gov snapshot contains 13,975 prepared rows after the 2015+ and comparable-fuel scope filters. Fixed seed-42 duplicate-grouped splits: 9,816 train, 2,080 validation, 2,079 previously used midterm holdout. Imputation, scaling and encoding fit only on training data. The holdout is not a new independent endterm test. EPA ratings are not a guarantee of real-world driving fuel use.

In [ ]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
if not (root / 'reports').exists(): root = root.parent.parent
results = root / 'reports/results/endterm'
comparison_val = pd.read_csv(results / 'model_comparison_validation.csv')
comparison_test = pd.read_csv(results / 'model_comparison_test.csv')
clusters = pd.read_csv(results / 'clustering_profiles.csv')
pca_variance = pd.read_csv(results / 'pca_variance.csv')
ann = json.loads((results / 'ann_metrics.json').read_text())
display(comparison_val)
display(clusters)
display(pca_variance)

## EDA, features and classical models

The preserved midterm EDA is under `notebooks/mid/`. Inputs are model year, engine specifications, drive, transmission, vehicle class, fuel type and the inherited displacement-per-cylinder feature. Make/model remain for identification and error analysis; MPG, row ID and target-derived variables are excluded. Classical tuning uses duplicate-grouped five-fold CV within train, followed by validation-MAE selection.

## K-Means engine profiles

K-Means uses only cylinders and displacement after training-only median imputation and StandardScaler. The retained solution has k=3: about 4 cylinders/2.04 L/27.64 MPG, 6 cylinders/3.30 L/21.02 MPG and 8.46 cylinders/5.22 L/16.72 MPG. Cluster IDs are labels, not quality ranks. Descriptive MPG differences are associations, not causal evidence.

![K-Means scatter](../../reports/figures/endterm/clustering_scatter.png)

## PCA and cluster overlay

PCA uses the same two engine features, fit on train only. PC1 retains 96.06% of variance; PC1 plus PC2 retains 100%. This is a rotation of a two-dimensional engine space, not a 2-to-2 reduction. PCA maximises variance, not MPG accuracy, and does not prove natural clusters.

![Explained variance](../../reports/figures/endterm/pca_explained_variance.png)

![PCA clusters](../../reports/figures/endterm/pca_clusters_train.png)

## Feedforward ANN

The ANN has 40 encoded inputs, hidden layers 64 and 32 with ReLU, and one linear MPG output. It uses Adam, learning rate 0.001, batch size 64, MSE on a train-standardised target, and early stopping. The restored best checkpoint is epoch 113; stopping occurred at epoch 133. Validation MAE is 1.082715 MPG, RMSE 1.522971 MPG and R-squared 0.939596. Validation selected the ANN configuration and checkpoint, so its metric is selection-biased.

![ANN learning curves](../../reports/figures/endterm/ann_learning_curves.png)

## Comparison and error analysis

The Decision Tree wins the predeclared validation-MAE rule: MAE 1.051024 MPG, RMSE 1.636483 MPG and R-squared 0.930256. ANN has a slightly higher MAE but lower RMSE and higher R-squared; this does not change the declared winner. On the previously used holdout, tree MAE is 1.073831 and ANN MAE is 1.078018 MPG.

Tree validation MAE is 1.827 MPG for 241 hybrid rows versus 0.949 MPG for 1,839 non-hybrid rows. This descriptive result needs caution because hybrid detail and other important vehicle variables are absent.

![Validation diagnostics](../../reports/figures/endterm/evaluation_validation.png)

## Conclusions, limitations and contributions

The endterm adds K-Means, PCA and ANN to the inherited regression workflow. The tree remains the primary-MAE winner; ANN is competitive. PCA shows one component captures most variation in the two selected engine features. Limits include EPA lab ratings, missing weight/horsepower/hybrid details, related variants across splits, one snapshot/seed, validation selection and prior holdout use.

Aktore: PCA and final integration. Khamza: ANN and reproducibility protocol. Askhat: K-Means, classical refinement, comparison and error analysis. Reproduce PCA with `py -3.12 -m vehicle_efficiency.dimensionality_reduction`; use the frozen comparison artifacts for reporting.